# Explorer les tables NeonDB

Objectif : se connecter à NeonDB depuis le `.env`, vérifier les tables disponibles, puis afficher un aperçu (`head`) de chaque table du projet Foot Predictor.

Le notebook ne modifie aucune donnée : il fait uniquement des lectures SQL.


## 1. Setup

Le notebook cherche une URL de connexion dans `NEON_DATABASE_URL`, `DATABASE_URL` ou `POSTGRES_URL`.


In [ ]:
from pathlib import Path
import os

import pandas as pd
from sqlalchemy import create_engine, text
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 50)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / ".env").exists():
    raise FileNotFoundError("Fichier .env introuvable dans foot-predictor.")

SCHEMA = "public"
TABLES = ["competition", "team", "player", "match", "match_team", "lineup", "lineup_audit"]
HEAD_ROWS = 5


## 2. Charger la connexion Neon

On lit le `.env` sans afficher la valeur de connexion.


In [ ]:
def load_env_file(path: Path) -> dict[str, str]:
    values = {}
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


def sqlalchemy_url(url: str) -> str:
    if url.startswith("postgres://"):
        return url.replace("postgres://", "postgresql+psycopg2://", 1)
    if url.startswith("postgresql://"):
        return url.replace("postgresql://", "postgresql+psycopg2://", 1)
    return url

local_env = load_env_file(PROJECT_ROOT / ".env")
DATABASE_URL = (
    os.environ.get("NEON_DATABASE_URL")
    or os.environ.get("DATABASE_URL")
    or os.environ.get("POSTGRES_URL")
    or local_env.get("NEON_DATABASE_URL")
    or local_env.get("DATABASE_URL")
    or local_env.get("POSTGRES_URL")
    or ""
)

if not DATABASE_URL:
    raise RuntimeError("Ajoute NEON_DATABASE_URL, DATABASE_URL ou POSTGRES_URL dans .env.")

engine = create_engine(sqlalchemy_url(DATABASE_URL), pool_pre_ping=True)
print("Connexion configuree :", bool(DATABASE_URL))


## 3. Résumé des tables

On vérifie le nombre de lignes et de colonnes avant les aperçus.


In [ ]:
summary_rows = []

with engine.begin() as connection:
    for table_name in TABLES:
        row_count = connection.execute(
            text(f'SELECT COUNT(*) FROM "{SCHEMA}"."{table_name}"')
        ).scalar_one()
        column_count = connection.execute(
            text("""
                SELECT COUNT(*)
                FROM information_schema.columns
                WHERE table_schema = :schema
                  AND table_name = :table_name
            """),
            {"schema": SCHEMA, "table_name": table_name},
        ).scalar_one()
        summary_rows.append({"table": table_name, "rows": row_count, "columns": column_count})

tables_summary = pd.DataFrame(summary_rows)
display(tables_summary)


## 4. Aperçu de chaque table

Chaque bloc charge seulement quelques lignes avec `LIMIT`, pas toute la table.


In [ ]:
heads = {}

for table_name in TABLES:
    query = f'SELECT * FROM "{SCHEMA}"."{table_name}" LIMIT {HEAD_ROWS}'
    heads[table_name] = pd.read_sql(query, engine)
    print(f"\n{table_name}")
    display(heads[table_name])


## 5. Exemple de jointure simple

Petit aperçu optionnel pour vérifier que `lineup` rejoint bien `match`, `team` et `player`.


In [ ]:
lineup_preview_query = f'''
SELECT
    l.match_id,
    m.match_date,
    ht.team_name AS home_team,
    at.team_name AS away_team,
    t.team_name AS lineup_team,
    mt.side,
    mt.score,
    mt.manager_name,
    mt.formation,
    l.lineup_type,
    l.position_player,
    l.minute_start,
    l.minute_end,
    l.minutes_played,
    l.player_id,
    p.player_name,
    p.sofifa_common_name,
    p.has_sofifa_profile
FROM "{SCHEMA}"."lineup" l
LEFT JOIN "{SCHEMA}"."match" m ON m.match_id = l.match_id
LEFT JOIN "{SCHEMA}"."match_team" mt ON mt.match_id = l.match_id AND mt.team_id = l.team_id
LEFT JOIN "{SCHEMA}"."team" t ON t.team_id = l.team_id
LEFT JOIN "{SCHEMA}"."match_team" hm ON hm.match_id = l.match_id AND hm.side = 'home'
LEFT JOIN "{SCHEMA}"."team" ht ON ht.team_id = hm.team_id
LEFT JOIN "{SCHEMA}"."match_team" am ON am.match_id = l.match_id AND am.side = 'away'
LEFT JOIN "{SCHEMA}"."team" at ON at.team_id = am.team_id
LEFT JOIN "{SCHEMA}"."player" p ON p.player_id = l.player_id
ORDER BY m.match_date DESC, l.match_id, mt.side, l.lineup_type, p.player_name
LIMIT {HEAD_ROWS * 4}
'''

display(pd.read_sql(lineup_preview_query, engine))

## 6. Notes

- `match` contient une ligne par match.
- `match_team` contient deux lignes par match : une par équipe, avec side, score, coach et formation.
- `lineup` contient une ligne par joueur présent sur la feuille de match, sans répéter les noms de match, équipe ou joueur.
- `lineup.minute_start`, `lineup.minute_end` et `lineup.minutes_played` décrivent le temps passé sur le terrain. Les remplaçants non utilisés ont `minutes_played = 0`.
- `player` contient toutes les données joueur disponibles, avec `has_sofifa_profile` pour distinguer les joueurs enrichis SoFIFA.